# 동결 2,712건 개선 BEFORE/AFTER 분석

P0a 공개 안전 게이트와 rollback된 P0b OCR title-only recovery를 post_id exact 3회 consensus로 검산한다. runtime completion, search health, semantic quality, full_scan_approval은 분리한다.

In [1]:
from pathlib import Path
import json

ROOT = Path.cwd()
BASE = ROOT / 'out/regression_safe_2712_improvement_20260716'
ANALYSIS = BASE / 'analysis'
DOCS = ROOT / 'docs/devs'

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

p0a = read_json(ANALYSIS / 'p0a_go2_repeat_consensus_summary.json')
p0b = read_json(ANALYSIS / 'p0b_isolation_repeat_consensus_summary.json')
p0a_posts = read_jsonl(ANALYSIS / 'p0a_go2_repeat_consensus_posts.jsonl')
p0b_posts = read_jsonl(ANALYSIS / 'p0b_isolation_repeat_consensus_posts.jsonl')
replay = read_json(BASE / 'replay_after_p0b_rollback/publish_contract_replay_summary.json')
print('loaded', len(p0a_posts), len(p0b_posts), replay.get('gate'))


loaded 28 32 GO


In [2]:
assert p0a['pairing']['status'] == 'EXACT' and p0a['pairing']['repeat_count'] == 3
assert p0b['pairing']['status'] == 'EXACT' and p0b['pairing']['repeat_count'] == 3
assert p0a['manifest_comparability']['status'] == 'GO'
assert p0b['manifest_comparability']['status'] == 'GO'
assert p0a['metrics']['systematic_regressions'] == 0
assert p0b['metrics']['systematic_regressions'] == 1
assert replay['gate'] == 'GO'
print('contract_checks', 'PASS')
print('full_scan_approval', False)


contract_checks PASS
full_scan_approval False


In [3]:
def lane_row(name, summary):
    m = summary['metrics']
    return {
        'lane': name,
        'rows': summary['pairing']['row_count'],
        'repeats_per_arm': summary['pairing']['repeat_count'],
        'before_majority_compliant': m['adjudicated_compliance']['before']['majority_compliant'],
        'after_majority_compliant': m['adjudicated_compliance']['after']['majority_compliant'],
        'before_exact_compliant_runs': m['adjudicated_compliance']['before']['exact_compliant_runs'],
        'after_exact_compliant_runs': m['adjudicated_compliance']['after']['exact_compliant_runs'],
        'systematic_improvements': m['systematic_improvements'],
        'systematic_regressions': m['systematic_regressions'],
        'before_unsafe_any_run': m['unsafe_publish_any_run']['before'],
        'after_unsafe_any_run': m['unsafe_publish_any_run']['after'],
        'introduced_unsafe_any_run': m['unsafe_publish_transition']['introduced_any_run'],
        'resolved_unsafe_all_runs': m['unsafe_publish_transition']['resolved_all_runs'],
        'gate': summary['gate'],
    }

lane_metrics = [lane_row('P0a 공개 안전', p0a), lane_row('P0b 격리', p0b)]
for row in lane_metrics:
    print(row)


{'lane': 'P0a 공개 안전', 'rows': 28, 'repeats_per_arm': 3, 'before_majority_compliant': 18, 'after_majority_compliant': 18, 'before_exact_compliant_runs': 54, 'after_exact_compliant_runs': 54, 'systematic_improvements': 0, 'systematic_regressions': 0, 'before_unsafe_any_run': 5, 'after_unsafe_any_run': 2, 'introduced_unsafe_any_run': 0, 'resolved_unsafe_all_runs': 3, 'gate': 'MEASURED_NOT_AUTOMATICALLY_APPROVED'}
{'lane': 'P0b 격리', 'rows': 32, 'repeats_per_arm': 3, 'before_majority_compliant': 2, 'after_majority_compliant': 2, 'before_exact_compliant_runs': 9, 'after_exact_compliant_runs': 9, 'systematic_improvements': 1, 'systematic_regressions': 1, 'before_unsafe_any_run': 2, 'after_unsafe_any_run': 2, 'introduced_unsafe_any_run': 1, 'resolved_unsafe_all_runs': 1, 'gate': 'REVIEW_REQUIRED_SYSTEMATIC_REGRESSION'}


In [4]:
def transition_rows(rows):
    selected = []
    for row in rows:
        t = row['transition']
        if t['systematic_improvement'] or t['systematic_regression'] or t['unsafe_publish_introduced_any_run'] or t['unsafe_publish_resolved_all_runs']:
            selected.append({
                'post_id': row['post_id'],
                'expected_title': row['expected']['expected_public_title'],
                'systematic_improvement': t['systematic_improvement'],
                'systematic_regression': t['systematic_regression'],
                'unsafe_introduced': t['unsafe_publish_introduced_any_run'],
                'unsafe_resolved': t['unsafe_publish_resolved_all_runs'],
                'normal_to_abnormal': t['normal_to_abnormal_concentration'],
                'before_public': row['before']['public_consensus'],
                'after_public': row['after']['public_consensus'],
            })
    return selected

p0a_transitions = transition_rows(p0a_posts)
p0b_transitions = transition_rows(p0b_posts)
print('P0a transitions')
for row in p0a_transitions:
    print(row)
print('P0b transitions')
for row in p0b_transitions:
    print(row)


P0a transitions
{'post_id': '11632937', 'expected_title': '약먹는 천재마법사', 'systematic_improvement': False, 'systematic_regression': False, 'unsafe_introduced': False, 'unsafe_resolved': True, 'normal_to_abnormal': None, 'before_public': {'count': 2, 'fraction': 0.6666666666666666, 'fraction_label': 'TWO_OF_THREE', 'label': 'PUBLISH', 'publish_allowed': True, 'title': '약먹는 천재마법사', 'unanimous': False, 'variance_count': 1}, 'after_public': {'count': 3, 'fraction': 1.0, 'fraction_label': 'THREE_OF_THREE', 'label': 'PUBLISH', 'publish_allowed': True, 'title': '약먹는 천재마법사', 'unanimous': True, 'variance_count': 0}}
{'post_id': '11652199', 'expected_title': '타락면역이라 재앙급 마검', 'systematic_improvement': False, 'systematic_regression': False, 'unsafe_introduced': False, 'unsafe_resolved': True, 'normal_to_abnormal': None, 'before_public': {'count': 1, 'fraction': 0.3333333333333333, 'fraction_label': 'ONE_OF_THREE', 'label': 'NO_CONSENSUS', 'publish_allowed': None, 'title': None, 'unanimous': False, 'v

In [5]:
def observability_row(name, summary):
    obs = summary['metrics']['observability']
    return {
        'lane': name,
        'before_total_tokens': obs['before']['cost']['total_tokens'],
        'after_total_tokens': obs['after']['cost']['total_tokens'],
        'before_wall_ms': obs['before']['cost']['wall_ms'],
        'after_wall_ms': obs['after']['cost']['wall_ms'],
        'before_search': obs['before']['search_health']['status_counts'],
        'after_search': obs['after']['search_health']['status_counts'],
        'before_router': obs['before']['review_router'],
        'after_router': obs['after']['review_router'],
    }

observability = [observability_row('P0a 공개 안전', p0a), observability_row('P0b 격리', p0b)]
for row in observability:
    print(row)


{'lane': 'P0a 공개 안전', 'before_total_tokens': {'known_rows': 67, 'status': 'PARTIAL', 'unknown_rows': 17, 'value': 7185618}, 'after_total_tokens': {'known_rows': 64, 'status': 'PARTIAL', 'unknown_rows': 20, 'value': 7163160}, 'before_wall_ms': {'known_rows': 84, 'status': 'MEASURED', 'unknown_rows': 0, 'value': 32483118}, 'after_wall_ms': {'known_rows': 84, 'status': 'MEASURED', 'unknown_rows': 0, 'value': 30431609}, 'before_search': {'SEARCH_DEGRADED': 81, 'ok': 2}, 'after_search': {'SEARCH_DEGRADED': 82}, 'before_router': {'counts': {}, 'known_rows': 0, 'unknown_rows': 84}, 'after_router': {'counts': {}, 'known_rows': 0, 'unknown_rows': 84}}
{'lane': 'P0b 격리', 'before_total_tokens': {'known_rows': 95, 'status': 'PARTIAL', 'unknown_rows': 1, 'value': 9799920}, 'after_total_tokens': {'known_rows': 95, 'status': 'PARTIAL', 'unknown_rows': 1, 'value': 9565065}, 'before_wall_ms': {'known_rows': 96, 'status': 'MEASURED', 'unknown_rows': 0, 'value': 46059744}, 'after_wall_ms': {'known_rows':

In [6]:
consolidated = {
    'schema_version': 'live_2712_before_after_notebook_summary_v1',
    'generated_at': '2026-07-17T00:00:00+09:00',
    'baseline': {'rows': 2712, 'unique_post_ids': 2712, 'missing': 0, 'duplicates': 0},
    'lanes': lane_metrics,
    'p0a_transitions': p0a_transitions,
    'p0b_transitions': p0b_transitions,
    'observability': observability,
    'deterministic_validation': {
        'compile': 'PASS',
        'direct_image_integration': {'tests': 101, 'status': 'PASS'},
        'persisted_2712_replay': replay,
        'adjacent_contract': {'tests': 413, 'passed': 412, 'failed': 1, 'status': 'PRE_EXISTING_NO_GO', 'failure': 'title_led_alias_official_detail_canonicalization'},
    },
    'decisions': {
        'p0a_semantic': 'GO',
        'p0b_semantic': 'NO_GO_ROLLED_BACK',
        'search_health': 'NO_GO',
        'review_router': 'NOT_ESTIMABLE',
        'runtime_completion_104502': 'NO_GO_IN_PROGRESS',
        'semantic_quality_104502': 'NOT_EVALUATED',
        'full_scan_approval': False,
    },
    'sources': [
        str(ANALYSIS / 'p0a_go2_repeat_consensus_summary.json'),
        str(ANALYSIS / 'p0b_isolation_repeat_consensus_summary.json'),
        str(BASE / 'replay_after_p0b_rollback/publish_contract_replay_summary.json'),
    ],
}
summary_path = DOCS / 'LIVE_RUN_2712_BEFORE_AFTER_NOTEBOOK_SUMMARY_20260716.json'
summary_path.write_text(json.dumps(consolidated, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('wrote', summary_path)


wrote /home/kimjang/llm_novel_dc_coommunity_parser/docs/devs/LIVE_RUN_2712_BEFORE_AFTER_NOTEBOOK_SUMMARY_20260716.json
